# ai_water: map surface water

Maps surface water with the trained model from Sentinel-1 on **any dates**, using both **ascending and descending** passes when the archive has them, together with the
site's **DEM** (slope, aspect), SDWI and land cover (context only). Radar shadow sits on different slopes in the two geometries, so the water **frequency** map is the
pixel-wise minimum of the two directions' frequencies.

| `RUN_MODE` | What it maps | How to choose the area |
|---|---|---|
| `"single"` | one water body | default **Angat Dam**; set `AOI` to the vector file of another dam / reservoir / lake / river |
| `"multiple"` | a list of water bodies in one run | `AOI` = a vector file with one feature per body (or a sites CSV); `AOI_NAMES` picks the ones to map |

The **whole country** is a separate job: run `scripts/map_nationwide.py` (tiles, pre-screening and a national index; see the README).

Per water body it writes the water mask of every date (optional), a **water-frequency** map (% of valid observations with water), the number of observations, and
one table of areas (`areas.csv`). A failing site is recorded and the run goes on; re-running resumes where it stopped.

Train a model first with `ai_water_training.ipynb` (it writes `models/terramind_water_decoder.pt`). Earth Engine is needed (DEM and land cover): set `GEE_PROJECT`.

## 0. Setup

In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path

# Colab: install the dependencies. numpy is pinned below 2.3 because Colab's preinstalled numba requires it; without the pin pip
# upgrades numpy and prints "numba requires numpy<2.3" (and numba then fails to import). If numpy was already imported in this
# session, restart the runtime once after this cell (Runtime > Restart session) and run the notebook again.
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2.3", "scipy", "scikit-image", "shapely", "pandas",
                    "matplotlib", "rasterio", "affine", "pyproj", "pystac-client", "planetary-computer", "odc-stac",
                    "earthengine-api", "torch", "terratorch", "geopandas"], check=True)

# repository root: hard-coded (edit this one line if the project lives elsewhere)
ROOT = Path("/content/drive/MyDrive/Surface_Water/project")
if "google.colab" in sys.modules and not ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")
assert (ROOT / "ai_water").is_dir(), f"ai_water/ not found in {ROOT}: check the path above and that Drive is mounted"
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

## 1. Configuration

In [ ]:
RUN_MODE = "single"                 # "single" | "multiple" (a list of bodies)
START, END = "2024-01-01", "2024-03-31"     # Sentinel-1 dates to map: any dates (the model needs no JRC)

# ---- model
THRESHOLD = 0.5                     # water probability at which a pixel is water
MAX_SLOPE_DEG = 25.0                # safeguard: water predicted on slopes steeper than this is removed and flagged (None = off)
USE_LANDCOVER = True                # False: blank the land-cover channels
MODEL = ROOT / "models" / "terramind_water_decoder.pt"      # small checkpoint (rebuilds the pretrained encoder from Hugging Face), or the
                                                            # full ROOT/"outputs"/"model"/"terramind_water.pt" (offline)

# ---- area
AOI = None                          # None = Angat Dam. Or a vector file of the water body: GeoJSON, or GeoPackage / Shapefile / KML; or a sites CSV
AOI_NAMES = None                    # multiple: only these features, e.g. ["Ambuklao Dam", "Binga Dam"]
NAME_COL = None                     # the property holding each feature's name (default: name / dam / label / id)
BUFFER_KM = 1.0                     # margin around each outline: shorelines keep their edge pixels
AOI_KIND = None                     # reservoir | lake | river (default: guessed from the geometry)

GEE_PROJECT = None                  # <-- your Google Cloud project id (always needed: Earth Engine supplies the DEM and land cover)

# ---- outputs
OUT = ROOT / "outputs" / "maps"
WRITE = ("water", "flags", "preview", "gpkg")     # per-date outputs: any of water, flags, preview, gpkg (polygons, water 1 / non-water 0), prob
COMPOSITE = True                    # frequency.tif / n_obs.tif per site
RESUME = True                       # False: reprocess dates already done
RES_M = 10.0
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rasterio

from ai_water import batch, infer, jrc, naming, scene, sites

assert RUN_MODE in ("single", "multiple"), "RUN_MODE must be 'single' or 'multiple' (the whole country: scripts/map_nationwide.py)"

## 2. Load the model

In [ ]:
model, meta, DEVICE = infer.load_model(MODEL)
print(f"model: {MODEL.name} on {DEVICE}")
print(f"trained on {meta.get('sites')}  ({meta.get('start')} .. {meta.get('end')}),  held out {meta.get('holdout')},  "
      f"best validation IoU vs labels {meta.get('best_val_iou_water')}")

## 3. The area to map

`single` and `multiple` use `sites.resolve_aoi`. With nothing set it returns **Angat Dam**. A vector file gives one site per feature; a file with several features is
only mapped whole when `RUN_MODE = "multiple"` (otherwise it must hold exactly one body, so a wrong file never silently maps the first feature).

In [ ]:
try:
    jrc.init_gee(GEE_PROJECT)
except Exception as e:
    print("Earth Engine init failed (", e.__class__.__name__, "), authenticating ...")
    jrc.init_gee(GEE_PROJECT, authenticate=True)

extra = {"buffer_km": BUFFER_KM, **{k: v for k, v in dict(name_col=NAME_COL, kind=AOI_KIND).items() if v}} if AOI and not str(AOI).lower().endswith(".csv") else {}
chosen = sites.resolve_aoi(AOI, AOI_NAMES, multiple=(RUN_MODE == "multiple"), **extra)

display(pd.DataFrame([{"site": s.label, "id": s.name, "kind": s.kind, "width_km": round(s.size_km()[0], 1), "height_km": round(s.size_km()[1], 1),
                       "crs": s.crs, "outline": s.geometry is not None} for s in chosen]))

fig, ax = plt.subplots(figsize=(6, 6))
for s in chosen:
    w, so, e, nn = s.bbox
    ax.plot([w, e, e, w, w], [so, so, nn, nn, so], lw=1.5)
    ax.annotate(s.label, (w, nn), fontsize=8)
ax.set_aspect("equal"); ax.set_xlabel("lon"); ax.set_ylabel("lat"); ax.set_title(f"{len(chosen)} area(s) to map ({RUN_MODE})")
plt.tight_layout(); plt.show()

## 4. Run

Reads Sentinel-1 RTC from Planetary Computer, one pass at a time. **Resumable**: dates already done are not read again, so after a disconnect just run this cell again.

In [ ]:
cfg = batch.MapConfig(start=START, end=END, out_dir=str(OUT), write=tuple(WRITE), composite=COMPOSITE, resume=RESUME,
                      cache_dir=str(OUT.parent / "cache"), device=DEVICE, threshold=THRESHOLD, max_slope_deg=MAX_SLOPE_DEG, use_landcover=USE_LANDCOVER,
                      scene=scene.SceneConfig())
result = batch.map_sites(chosen, model, cfg)

## 5. Results

The table has one row per water body, date and pass direction: the water area (km2), the area the terrain safeguard removed (water predicted on ground steeper than `MAX_SLOPE_DEG`:
candidates for radar shadow or layover), and the observed area. **Failures** (a site no pass covers, a read error) are listed, and the others are unaffected.

In [ ]:
areas = result.areas
if result.failures:
    print(f"{len(result.failures)} failure(s):")
    display(pd.DataFrame(result.failures))
assert len(areas), "nothing was mapped: read the output of the run cell above (dates, area, Planetary Computer errors)"
display(areas.round(3).tail(30))
print(areas.groupby("site")["water_km2"].agg(["count", "mean", "min", "max"]).round(3).to_string())

### Water area through time

In [ ]:
d = areas.assign(date=pd.to_datetime(areas["date"]))
fig, ax = plt.subplots(figsize=(9, 3.6))
for (s, o), g in d.groupby(["label", "orbit"]):
    ax.plot(g["date"], g["water_km2"], marker=("^" if o == "ascending" else "v"), label=f"{s} {o}")
ax.set_ylabel("water area (km2)"); ax.set_title("water area (^ ascending, v descending)"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### Maps

The latest per-date water mask (if written) and the **water frequency** (% of valid observations with water: permanent water is 100, seasonal or flooded areas are in between).
With both pass directions the frequency is the minimum of the two, so radar shadow (water-like in one geometry only) drops out; the frequency figure (one per water body, both directions combined) is the second panel; compare `frequency_ascending.tif` and `frequency_descending.tif` in the output folder.

In [ ]:
for site in chosen[:4]:
    sdir = OUT / site.name
    rows = areas[areas["site"] == site.name]
    if not len(rows):
        continue
    last, orbit = rows.sort_values("date").iloc[-1][["date", "orbit"]]
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    mask = sdir / naming.scene_name(last, "water")
    if mask.exists():
        with rasterio.open(mask) as r:
            ax[0].imshow(r.read(1), cmap="Blues", interpolation="nearest")
        ax[0].set_title(f"{site.label}: water on {last} ({orbit})")
    else:
        ax[0].text(0.5, 0.5, "per-date mask not written (WRITE)", ha="center"); ax[0].set_title(site.label)
    pngs = sorted(sdir.glob("*surfacewaterfreq.png"))
    if pngs:
        ax[1].imshow(plt.imread(pngs[-1])); ax[1].set_title(pngs[-1].name, fontsize=7)
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

## Outputs

Everything is under `OUT` (`outputs/maps` on Drive): `areas.csv`, `failures.csv`, and per site folder `frequency.tif` (the minimum over the two pass directions),
`frequency_ascending.tif` and `frequency_descending.tif` (when both exist), `n_obs.tif`, and the named products (`ai_water/naming.py`):

- `philsa_multimodal-surfacewater-weekly_YYYYMMDD_v0.1.0_r01_surfacewater.tif`, `..._flags.tif`, `..._preview.png` - one set per mapped date;
- `philsa_multimodal-surfacewaterfreq-frequency_YYYYMM-YYYYMM_v0.1.0_r01_surfacewaterfreq.png` - the frequency picture over the months covered.

The whole country is mapped by `scripts/map_nationwide.py`, not here.

- `flags` marks `2` where the model said water but the slope is steeper than `MAX_SLOPE_DEG`: radar shadow or layover. Look at them before trusting an area.
- If a map still shows water streaks on slopes, compare `frequency_ascending.tif` with `frequency_descending.tif`: a streak that is bright in only one of them is shadow, and
  more training sites on steep terrain (rows in `data/training_sites.csv`) will teach the model to reject it.
- Rivers a few pixels wide are mostly mixed pixels at 10 m and are the weakest case.